In [43]:
# source ../.venv/bin/activate

# trzeba uruchomic jupyter lab z tego juz dzialajacego srodowiska


In [ ]:
import numpy as np
import math
from scipy.optimize import minimize_scalar


def fit_alpha_mle(m, m1, m2):
    """
    MLE dla nachylenia RGB luminosity function:

        phi(m) ~ 10**(alpha * (m - m1))

    w zakresie m1 <= m <= m2.

    Zwraca:
        alpha_mle, alpha_err, result
    """

    m = np.asarray(m, dtype=float)

    # wybór gwiazd w zakresie fitu
    mask = (m >= m1) & (m <= m2)
    mm = m[mask]

    N = len(mm)
    if N < 3:
        raise ValueError("Za mało gwiazd w zakresie fitu.")

    W = m2 - m1
    x = mm - m1
    ln10 = math.log(10.0)

    def loglike(alpha):
        """
        log L(alpha)
        """
        if alpha <= 0:
            return -np.inf

        lam = alpha * ln10

        return (
            N * math.log(lam)
            + lam * np.sum(x)
            - N * math.log(math.exp(lam * W) - 1.0)
        )

    def neg_loglike(alpha):
        return -loglike(alpha)

    res = minimize_scalar(
        neg_loglike,
        bounds=(0.001, 2.0),
        method="bounded"
    )

    alpha_mle = res.x

    # analityczna druga pochodna log-likelihood
    lam = alpha_mle * ln10
    E = math.exp(lam * W)

    # d2 lnL / d alpha2
    d2logL = (
        -N / alpha_mle**2
        + N * (ln10**2) * (W**2) * E / (E - 1.0)**2
    )

    alpha_err = math.sqrt(-1.0 / d2logL)

    return alpha_mle, alpha_err, res

# overlaping moving window diagnostics

def local_alpha_ratio(m, d=1.0, step=0.01, mmin=None, mmax=None):

    m = np.asarray(m)

    if mmin is None:
        mmin = m.min() + d
    if mmax is None:
        mmax = m.max() - d

    x = np.arange(mmin, mmax, step)

    alpha = []
    sigma_alpha = []

    for z in x:

        a = np.sum((m > z-d) & (m < z))
        b = np.sum((m > z) & (m < z+d))

        if a > 0 and b > 0:

            alpha_i = np.log10(b/a) / d

            sigma_i = (
                1.0 / (d * np.log(10))
                * np.sqrt(1.0/a + 1.0/b)
            )

        else:
            alpha_i = np.nan
            sigma_i = np.nan

        alpha.append(alpha_i)
        sigma_alpha.append(sigma_i)

    return x, np.array(alpha), np.array(sigma_alpha)

In [ ]:
alpha_mle, alpha_err, res = fit_alpha_mle(m,m1=m_trgb + 0.5,m2=m_trgb + 2.0)

print(f"alpha = {alpha:.4f} ± {alpha_err:.4f}")

x_alpha, alpha_loc, sigma_alpha = local_alpha_ratio(m,d=0.5,step=0.01)



print(f"alpha local = {np.nanmean(alpha_loc):.4f} ± {np.nanstd(alpha_loc):.4f}")

In [ ]:
mask = (
    np.isfinite(alpha_loc)
    & np.isfinite(sigma_alpha)
)

# obserwowany RMS wokół globalnego MLE
R_obs2 = np.mean(
    (alpha_loc[mask] - alpha_mle)**2
)

# oczekiwany wkład szumu Poissona
R_noise2 = np.mean(
    sigma_alpha[mask]**2
)

# nadmiarowa "chropowatość"
R_excess2 = max(0.0, R_obs2 - R_noise2)
R_excess = np.sqrt(R_excess2)

print(f"R_obs    = {np.sqrt(R_obs2):.4f}")
print(f"R_noise  = {np.sqrt(R_noise2):.4f}")
print(f"R_excess = {R_excess:.4f}")

In [ ]:


fig, ax1 = plt.subplots()

# ---------- LF ----------
ax1.hist(
    m,
    bins=30,
    histtype='bar',
    alpha=0.6,
    label='LF'
)

ax1.axvline(
    m_trgb,
    color='red',
    linestyle='--',
    label='TRGB'
)

ax1.set_yscale('log')
ax1.invert_xaxis()

ax1.set_xlabel("magnitude")
ax1.set_ylabel("log(N)")


# ---------- local alpha ----------
ax2 = ax1.twinx()

ax2.plot(
    x_alpha,
    alpha_loc,
    '.-',
    markersize=3,
    label=r'local $\alpha$'
)

# prawdziwa wartość z symulacji
ax2.axhline(
    0.3,
    color='red',
    linestyle='--',
    label=r'true $\alpha=0.3$'
)

# wartość MLE
ax2.axhline(
    alpha_mle,
    linestyle='-',
    linewidth=2,
    label=rf'MLE $\alpha={alpha_mle:.3f}\pm{alpha_err:.3f}$'
)

# niepewność MLE
ax2.axhspan(
    alpha_mle - alpha_err,
    alpha_mle + alpha_err,
    alpha=0.15
)

ax2.set_ylabel(r'local $\alpha$')

plt.title("RGB Luminosity Function")
plt.show()

In [ ]:

#=================== FILTR POISSON NOISE =======================================  

#x,f=poisson(m,1,["step=0.01","sigma=0.05"])
def poisson(m,d,params):

    step=0.01
    sigma=0.00
    mmin = m.min()
    mmax = m.max()-d
    if len(params)>0:
       for par in params:
    	   if "step=" in par: 
               step=float(par.split("=")[1])
               if "sigma=" in par: 
                   sigma=float(par.split("=")[1])
    ########
    if sigma>0:
       gauss_x=np.arange(-5*sigma,5*sigma,step,dtype=float)
       gaussian = lambda x,sigma: 1.*np.exp(-x**2/(2.*sigma**2))
       gauss_y=gaussian(gauss_x,sigma)
       gauss_y=gauss_y/np.sum(gauss_y)
    #########  
    x=np.arange(mmin,mmax,step,dtype=float)  
    f=[]
    
    for i,cos in enumerate(x):
       z=x[i]
    
       maska=m<z
       m1=m[maska]
       maska=m1>z-d
       m1=m1[maska]
       a=len(m1) 

       maska=m>z
       m2=m[maska]
       maska=m2<z+d
       m2=m2[maska]
       b=len(m2)
    
       dN=(b-a)*(b-a)
       if a==0: a=0.1
       if b==0: b=0.1
       tmp=(dN)/(a/2.+b/2.)
       tmp=tmp**0.5
       f=np.append(f,tmp )
###############
    if sigma>0:
       f=np.convolve(gauss_y,f,"same")    
###############   
    return x,f
  
#=============== detect ====================
# trgb,err1,err2=PoissonDetect(x,chi,trgb0)
def PoissonDetect(x,chi,trgb0):
    if trgb0>90 or trgb0==0:
        i=np.argmax(chi)
        trgb=x[i]
    else:
        x=np.array(x)
        tmp=(x-trgb0)
        tmp=tmp*tmp
        i=np.argmin(tmp)
        k=i
        l=i
        x0=x[i]
        zmiana=1
        while zmiana==1 and k+1<len(x) and l-1>0:
            if chi[k] > chi[k+1] and chi[k]>chi[k-1]: 
                zmiana=0
                trgb=x[k]
            elif chi[l] > chi[l+1] and chi[l]>chi[l-1]: 
                zmiana=0
                trgb=x[l]
            else: 
                zmiana=1   
            k=k+1
            l=l-1
    return trgb


In [ ]:
x,f = poisson(m,0.1,["step=0.01","sigma=0.05"])
trgb = PoissonDetect(x,f,0)
print(trgb)

In [ ]:
fig, ax1 = plt.subplots()
h = ax1.hist(m, bins=30, histtype='step', color='black', label='LF')

ax2 = ax1.twinx()
l1, = ax2.plot(x, f, '-b', label='Poisson')
l2 = ax1.axvline(m_trgb, color='red', linestyle='--', label='TRGB')

ax1.invert_xaxis()
handles = [l1, l2]
labels = [h[2][0].get_label(), 'Poisson', 'TRGB']
ax1.legend([h[2][0], l1, l2], labels)

plt.show()

In [ ]:

  
#=================== FILTR SOBELA =======================================  
  
def make_sobel(sigma,j,logscale=False):
    numerical_step=0.0005
    m=np.arange(j.min(),j.max()+numerical_step,numerical_step,dtype=float)
    x= (j-j.min())/numerical_step
    indeks = x.astype(int)
    a=np.zeros(len(m))
    for i in indeks:
        a[i]=a[i]+1 
    gauss_x=np.arange(-5*sigma,5*sigma,numerical_step,dtype=float)
    gaussian = lambda x,sigma: 1.*np.exp(-x**2/(2.*sigma**2))
    gauss_y=gaussian(gauss_x,sigma) 
    splot=np.convolve(a,gauss_y,"same")    #to zajmuje mase czasu
    miu=int(0.5*sigma/numerical_step)
    pomocnicza1=splot[2*miu:len(splot)]
    pomocnicza2=splot[0:len(splot)-2*miu]
    maska=pomocnicza1<0.01
    pomocnicza1[maska]=0.01
    maska=pomocnicza2<0.01
    pomocnicza2[maska]=0.01
    sobel_x=m[miu:len(splot)-miu]
    if logscale: 
        sobel=np.log10(pomocnicza1)-np.log10(pomocnicza2)
    else: 
        sobel=pomocnicza1-pomocnicza2
    
    sobel=sobel*splot.max()/sobel.max()
    maska=sobel<0
    sobel[maska]=0
    trim = int((len(sobel) - len(sobel_x))/2.)
    if trim > 0 : 
        sobel=sobel[trim:np.alen(sobel)-trim]
    return m,splot,sobel_x,sobel  
  
  
def sobel_detect(x,sobel,trgb0):
    
    Nmax=max(x)
    trgb=9.
    if trgb0>90 or trgb0==0:
        i=np.argmax(sobel)
        trgb=x[i]
    else:
        x=np.array(x)
        tmp=(x-trgb0)
        tmp=tmp*tmp
        i=np.argmin(tmp)
        k=i
        l=i
        x0=sobel[i]
        zmiana=1
        while zmiana==1 and k+1<len(sobel) and l-1>0:
            if sobel[k] > sobel[k+1] and sobel[k]>sobel[k-1]: 
                zmiana=0
                trgb=x[k]
            elif sobel[l] > sobel[l+1] and sobel[l]>sobel[l-1]: 
                zmiana=0
                trgb=x[l]
            else: 
                zmiana=1   
            k=k+1
            l=l-1
    return trgb      
 

In [ ]:
x,l,xf,f = make_sobel(0.03,m,False)
trgb = sobel_detect(x,f,10)
print(trgb)

In [ ]:
fig, ax1 = plt.subplots()
h = ax1.hist(m, bins=30, histtype='step', color='black', label='LF')

ax2 = ax1.twinx()
l0, = ax2.plot(x, l, '-g', label='Poisson')
l1, = ax2.plot(xf, f, '-b', label='Poisson')
l2 = ax1.axvline(m_trgb, color='red', linestyle='--', label='TRGB')

ax1.invert_xaxis()
handles = [l1, l2]
labels = [h[2][0].get_label(), 'Poisson', 'TRGB']
ax1.legend([h[2][0], l1, l2], labels)

plt.show()

In [ ]:

  
#=================== MLA ===========================================
#trgb,a,b,c=mla([10,0.3,1,0.3],0.01,rozklad)  tak jak w pracy Makarov et al. 2006
#sigma_mla to jest rozmycie fi w zwiazku z bledami fot.
#ten pierwszy array to wartosci poczatkowe
#
#x,y=fi_mla(x,trgb,a,b,c,sigma_mla)
#

from scipy.optimize import minimize


def mla(p0,sigma,gtol,rozklad,trgb0):

    res = minimize(
        wiarygodnosc,
        p0,
        args=(rozklad,sigma),
        method='SLSQP',
        bounds=[
            (trgb0-0.5,trgb0+0.5),
            (-1,1),
            (-3,3),
            (-1,1)
        ],
        tol=gtol
    )

    return res.x


def wiarygodnosc(p, r, sigma):

    trgb, a, b, c = p

    _, A = fi_mla(r, trgb, a, b, c, sigma)

    A = np.maximum(A, 1e-300)

    x = np.arange(np.min(r), np.max(r), 0.01)

    _, y = fi_mla(x, trgb, a, b, c, sigma)

    B = np.log(np.sum(y))

    L = -np.sum(np.log(A)) + len(r) * B

    return L


def fi_mla(r, trgb, a, b, c, sigma):

    x = np.arange(np.min(r), np.max(r), 0.01)

    x1 = x[x > trgb]
    y1 = np.exp(np.clip(np.log(10)*(a*(x1-trgb)+b),-700,700))

    x2 = x[x < trgb]
    y2 = np.exp(np.clip(np.log(10)*(c*(x2-trgb)),-700,700))

    xk = np.concatenate((x1, x2))
    yk = np.concatenate((y1, y2))

    idx = np.argsort(xk)

    xk = xk[idx]
    yk = yk[idx]

    if sigma > 0:

        gauss_x = np.arange(-3*sigma, 3*sigma, 0.01)

        gauss_y = np.exp(
            -gauss_x**2/(2*sigma**2)
        )

        gauss_y /= np.sum(gauss_y)

        yk = np.convolve(
            yk,
            gauss_y,
            mode="same"
        )

    y_interp = np.interp(r, xk, yk)

    return r, y_interp
  

In [ ]:
# początkowe zgadywanie
p = mla(
    [10.0,0.3,1,0.3],
    0.05,
    1e-6,
    m,
    trgb0=10.02
)

trgb, a, b, c = p

print("TRGB =", trgb)
print("a =", a)
print("b =", b)
print("c =", c)


# model LF
x_model = np.linspace(
    np.min(m),
    np.max(m),
    500
)

_, y_model = fi_mla(
    x_model,
    trgb,
    a,
    b,
    c,
    0.05
)


plt.figure(figsize=(8,5))

plt.hist(
    m,
    bins=30,
    histtype='step',
    label="LF"
)

# skalowanie modelu
hist_max = np.histogram(m,bins=30)[0].max()

y_model = y_model * hist_max / y_model.max()

plt.plot(
    x_model,
    y_model,
    '-r',
    linewidth=2,
    label="MLA"
)

plt.axvline(
    trgb,
    color='blue',
    linestyle='--',
    label=f"TRGB={trgb:.3f}"
)

plt.gca().invert_xaxis()

plt.xlabel("Magnitude")
plt.ylabel("N")
plt.legend()
plt.title("TRGB Maximum Likelihood Fit")

plt.show()

In [ ]:

def gloess(x, y, sigma):

    """
    Gaussian-windowed locally weighted scatterplot smoothing

    x     - bin centers / magnitudes
    y     - luminosity function
    sigma - smoothing scale (mag)
    """

    y_smooth = np.zeros_like(y, dtype=float)

    for i, xi in enumerate(x):

        # odległość od aktualnego punktu
        dx = x - xi

        # Gaussian weights
        w = np.exp(
            -0.5*(dx/sigma)**2
        )

        # lokalna średnia ważona
        y_smooth[i] = np.sum(w*y) / np.sum(w)

    return y_smooth

def gloess_linear(x,y,sigma):

    ys = np.zeros_like(y,dtype=float)

    for i,xi in enumerate(x):

        dx = x-xi

        w = np.exp(
            -0.5*(dx/sigma)**2
        )


        X = np.vstack(
            [
                np.ones(len(x)),
                dx
            ]
        ).T


        W = np.diag(w)


        beta = np.linalg.solve(
            X.T @ W @ X,
            X.T @ W @ y
        )


        ys[i] = beta[0]


    return ys

def gloess_edge(x,y,sigma):

    smooth = gloess_linear(
        x,y,sigma
    )

    edge = np.gradient(
        smooth,
        x
    )

    return smooth, edge    


In [ ]:
hist, edges = np.histogram(
    m,
    bins=100
)

x = 0.5*(edges[1:]+edges[:-1])
y = hist

smooth, edge = gloess_edge(
    x,
    y,
    sigma=0.05
)


i = np.argmax(edge)

trgb_gloess = sobel_detect(x,edge,10)

print("TRGB GLOESS =", trgb_gloess)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np




fig, ax1 = plt.subplots()

ax2 = ax1.twinx()



# --- LF ---
ax1.step(
    x,
    y,
    where="mid",
    label="LF"
)

ax2.plot(
    x,
    smooth,
    "-r",
    lw=2,
    label="GLOESS"
)

ax2.axvline(
    trgb_gloess,
    color="blue",
    ls="--",
    label=f"TRGB={trgb_gloess:.3f}"
)

ax1.invert_xaxis()
ax1.set_ylabel("N")
ax1.legend()


# --- EDGE ---
ax2.plot(
    x,
    edge,
    "-k",
    lw=2,
    label="d(GLOESS)/dm"
)



plt.tight_layout()
plt.show()

pik musi byc wysoki i samotny

statystyki S1 = max(chi)

R21 = S1 / S2 (1mag)

dm21 = abs(m1-m2)

N_07max_1mag = liczba pikow w jednen mag o wysokosci odpowiednio 0.8max, 0.7max, 0.5max

lokalna dominacja: D = (chi1-chi2)/chi1 = 1 - chi1/chi2

lokalna chropowatosc: Q = liczba maksimow w odleglosci d od centrum.